## STEP 1 — Generate the IoT data in Google **Colab** **bold text**

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)

# Chennai zones
zones = ['Anna Nagar', 'T Nagar', 'Velachery', 'Adyar',
         'Tambaram', 'Porur', 'OMR', 'Chromepet',
         'Mylapore', 'Ambattur']

# Base coordinates for Chennai
zone_coords = {
    'Anna Nagar':  (13.0850, 80.2101),
    'T Nagar':     (13.0418, 80.2341),
    'Velachery':   (12.9815, 80.2180),
    'Adyar':       (13.0012, 80.2565),
    'Tambaram':    (12.9249, 80.1000),
    'Porur':       (13.0359, 80.1567),
    'OMR':         (12.9010, 80.2279),
    'Chromepet':   (12.9516, 80.1462),
    'Mylapore':    (13.0368, 80.2676),
    'Ambattur':    (13.1143, 80.1548)
}

vehicle_types = ['Bike', 'Scooter', 'Cycle']
statuses = ['Active', 'Idle', 'Speeding', 'Delayed']

records = []

for vehicle_id in range(1, 501):  # 500 vehicles
    zone = random.choice(zones)
    base_lat, base_lon = zone_coords[zone]
    vehicle_type = random.choice(vehicle_types)
    agent_name = f"Agent_{vehicle_id:03d}"

    for trip in range(random.randint(5, 15)):  # 5-15 trips per agent
        hour = random.randint(8, 23)
        minute = random.randint(0, 59)
        timestamp = datetime(2024, random.randint(1,3),
                           random.randint(1,28), hour, minute)

        speed = round(np.random.normal(30, 12), 1)
        speed = max(0, min(80, speed))

        distance = round(random.uniform(0.5, 15), 2)
        idle_time = round(random.uniform(0, 20), 1)
        delivery_time = round(random.uniform(10, 60), 1)

        if speed > 60:
            status = 'Speeding'
        elif idle_time > 15:
            status = 'Idle'
        elif delivery_time > 45:
            status = 'Delayed'
        else:
            status = 'Active'

        lat = base_lat + np.random.uniform(-0.05, 0.05)
        lon = base_lon + np.random.uniform(-0.05, 0.05)

        records.append({
            'vehicle_id': f'VH{vehicle_id:03d}',
            'agent_name': agent_name,
            'vehicle_type': vehicle_type,
            'zone': zone,
            'timestamp': timestamp,
            'hour': hour,
            'speed_kmh': speed,
            'distance_km': distance,
            'idle_time_min': idle_time,
            'delivery_time_min': delivery_time,
            'status': status,
            'latitude': round(lat, 6),
            'longitude': round(lon, 6)
        })

df = pd.DataFrame(records)
print(f"Generated {len(df):,} trip records ✅")
print(df['status'].value_counts())
print(df.head())

Generated 4,924 trip records ✅
status
Active      2637
Idle        1210
Delayed     1049
Speeding      28
Name: count, dtype: int64
  vehicle_id agent_name vehicle_type      zone           timestamp  hour  \
0      VH001  Agent_001         Bike  Tambaram 2024-03-08 10:10:00    10   
1      VH001  Agent_001         Bike  Tambaram 2024-02-07 18:05:00    18   
2      VH001  Agent_001         Bike  Tambaram 2024-02-25 15:48:00    15   
3      VH001  Agent_001         Bike  Tambaram 2024-01-02 10:44:00    10   
4      VH001  Agent_001         Bike  Tambaram 2024-01-13 20:54:00    20   

   speed_kmh  distance_km  idle_time_min  delivery_time_min   status  \
0       36.0         1.47            9.7               16.7   Active   
1       28.3        14.53            2.2               58.5  Delayed   
2       49.0         3.61           10.4               42.6   Active   
3       39.2         5.49           16.2               15.2     Idle   
4       24.4         5.82            7.5           

STEP 2 — Run SQL **analysis** **bold text**

In [2]:
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('vehicle_trips', conn, index=False, if_exists='replace')

print("\n--- Q1: Which zones have most delays? ---")
q1 = pd.read_sql("""
    SELECT zone,
           COUNT(*) as total_trips,
           SUM(CASE WHEN status='Delayed' THEN 1 ELSE 0 END) as delayed_trips,
           ROUND(AVG(delivery_time_min),1) as avg_delivery_min
    FROM vehicle_trips
    GROUP BY zone
    ORDER BY delayed_trips DESC
""", conn)
print(q1)

print("\n--- Q2: Peak hours of delivery activity ---")
q2 = pd.read_sql("""
    SELECT hour,
           COUNT(*) as total_trips,
           ROUND(AVG(speed_kmh),1) as avg_speed
    FROM vehicle_trips
    GROUP BY hour
    ORDER BY hour
""", conn)
print(q2)

print("\n--- Q3: Vehicles that are speeding most ---")
q3 = pd.read_sql("""
    SELECT vehicle_id, agent_name, vehicle_type,
           COUNT(*) as speeding_count,
           ROUND(MAX(speed_kmh),1) as max_speed
    FROM vehicle_trips
    WHERE status = 'Speeding'
    GROUP BY vehicle_id
    ORDER BY speeding_count DESC
    LIMIT 10
""", conn)
print(q3)

print("\n--- Q4: Vehicles needing maintenance (>100km total) ---")
q4 = pd.read_sql("""
    SELECT vehicle_id, agent_name,
           ROUND(SUM(distance_km),1) as total_distance_km,
           COUNT(*) as total_trips
    FROM vehicle_trips
    GROUP BY vehicle_id
    HAVING total_distance_km > 100
    ORDER BY total_distance_km DESC
    LIMIT 10
""", conn)
print(q4)

print("\nSQL Analysis done ✅")


--- Q1: Which zones have most delays? ---
         zone  total_trips  delayed_trips  avg_delivery_min
0    Tambaram          565            126              34.6
1     T Nagar          558            126              35.5
2   Velachery          542            122              35.0
3       Porur          547            109              34.7
4  Anna Nagar          504            108              34.4
5       Adyar          449             99              34.0
6         OMR          444             92              34.7
7    Ambattur          455             91              33.5
8    Mylapore          415             89              34.7
9   Chromepet          445             87              34.7

--- Q2: Peak hours of delivery activity ---
    hour  total_trips  avg_speed
0      8          309       30.3
1      9          292       30.3
2     10          296       30.1
3     11          309       29.4
4     12          310       29.4
5     13          312       30.2
6     14          279

STEP 3 — Export for Power **BI** **bold text**

In [4]:
# Main dataset
df.to_csv("vehicle_tracking.csv", index=False)

# SQL result tables
q1.to_csv("zone_delays.csv", index=False)
q2.to_csv("peak_hours.csv", index=False)
q3.to_csv("speeding_agents.csv", index=False)
q4.to_csv("maintenance_needed.csv", index=False)

from google.colab import files
files.download("vehicle_tracking.csv")
files.download("zone_delays.csv")
files.download("peak_hours.csv")
files.download("speeding_agents.csv")
files.download("maintenance_needed.csv")

print("All files downloaded ✅")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All files downloaded ✅
